In [1]:
# ----------- Science Jubilee -------------
from science_jubilee import Machine as Jub
from science_jubilee.tools import Pipette
import time

import numpy as np
import pandas as pd
import os

In [2]:
# defines the jubilee to call
jubilee = Jub.Machine(address='192.168.1.2', simulated = True)

sending: G90


In [3]:
# load deck
deck = jubilee.load_deck('lab_automation_deck.json')

# this loads the well plate for the destination for samples (sample_holder_name, position)
samples = jubilee.load_labware('septavialrev1_44_holder_2000ul.json', 2) # will change this
# this line of code defines the boundary conditions of the well plate requires 3 points
upper_left = (19.9,183.2)
upper_right = (132.4, 183.4)
lower_right = (132.5, 113.0)
samples.manual_offset([upper_left,upper_right,lower_right])
samples.load_manualOffset()

New manual offset applied to septavialrev1_44_holder_2000ul
New manual offset applied to septavialrev1_44_holder_2000ul


In [4]:
# this is same as lines above just for stock solution
stocks = jubilee.load_labware('20mlscintillation_12_wellplate_18000ul.json', 3)
upper_left = (167.4,176.8)
upper_right = (258.0, 176.8)
lower_right = (258.0, 119.8)
stocks.manual_offset([upper_left,upper_right,lower_right])
stocks.load_manualOffset()

New manual offset applied to 20mlscintillation_12_wellplate_18000ul
New manual offset applied to 20mlscintillation_12_wellplate_18000ul


In [5]:
# same as above just for pipette tip rack
tiprack = jubilee.load_labware('opentrons_96_tiprack_300ul.json', 0)
upper_left = (27.0,82.7)
upper_right = (125.6,82.7)
lower_right = (125.8,19.4)
tiprack.manual_offset([upper_left,upper_right,lower_right])
tiprack.load_manualOffset()

New manual offset applied to opentrons_96_tiprack_300ul
New manual offset applied to opentrons_96_tiprack_300ul


In [6]:
# defines position of trash can for tips
trash = jubilee.load_labware('agilent_1_reservoir_290ml.json', 1)

In [7]:
deck.safe_z

64.49

### LOAD TOOLS

In [8]:
P300 = Pipette.Pipette.from_config(3,'Pipette','P300_config.json')
jubilee.load_tool(P300)

P300.add_tiprack(tiprack)
P300.trash = trash[0]

sending: G90
sending: G0     V310.00 F2500.00 
sending: M400
sending: M409 K"tools"


TypeError: the JSON object must be str, bytes or bytearray, not NoneType

### EXPERIMENT FUNCTIONS

In [61]:
# lists are from a specific excel workbook since specefic case
df = pd.read_csv("co_volumes.csv")
co = df.to_numpy()
df2 = pd.read_csv("po_volumes.csv")
po = df2.to_numpy()
df3 = pd.read_csv("h20_volumes.csv")
h20 = df3.to_numpy()

In [ ]:
# these are the values that you will have to define and know beforehand to operate
#co = []
#po = [] #2D array of volumes, length of list will be the number of additions
#h20 = []

n_additions = 0 #how many times each sample will recieve new material
n_samples = 0 #how many samples created

In [20]:
def transfer_small(v, stock, target_sample):
    P300.aspirate(v, stocks[stock].bottom(+3))
    P300.dispense(v, samples[target_sample].bottom(+25))
    P300.blowout()

In [21]:
def transfer_large(v, stock, target_sample):
    if v > 300 & v <= 600:
        half = v / 2.0
        for k in range(2):
            P300.aspirate(half, stocks[stock].bottom(+3))
            P300.dispense(half, samples[target_sample].bottom(+25))
            P300.blowout()
    else:
        half = v / 3.0
        for i in range(3):
            P300.aspirate(half, stocks[stock].bottom(+3))
            P300.dispense(half, samples[target_sample].bottom(+25))
            P300.blowout()

In [19]:
def cobalt_addition(l):
    P300.pickup_tip("A2")
    for i in range(len(l)):
        if l[i] <= 300:
            transfer_small(l[i],1,i) # l is a list of volumes, each sample in order from 0 to i will have its own volume
        elif l[i] > 300:
            transfer_large(l[i],1,i)
    P300.return_tip("A2")

In [17]:
def phosphate_addition(l):
    P300.pickup_tip("A3")
    for i in range(len(l)):
        if l[i] <= 300:
            transfer_small(l[i],2,i)
        elif l[i] > 300:
            transfer_large(l[i],2,i)
    P300.return_tip("A3")

In [18]:
def water_addition(l):
    P300.pickup_tip("A1")
    for i in range(len(l)):
        if l[i] <= 300:
            transfer_small(l[i],0,i)
        elif l[i] > 300:
            transfer_large(l[i],0,i)
    P300.return_tip("A1")

In [62]:
def run_time_step(addition_step):
    # first begin with adding cobalt, co is a list of volumes to pipet
    cobalt_addition(co[addition_step])
    water_addition(h20[addition_step])
    phosphate_addition(po[addition_step])

In [64]:
def run_experiment(n_additions):
        # --- 1. START: log timestamp before any work begins ---
    start_time = time.time()
    log = []  # store all timestamp entries
    log.append({
        "event": "start",
        "timestamp": start_time,
        "message": f"Function started at {start_time.isoformat()}"
    })
    
    for k, step_number in enumerate(n_additions):
        run_time_step(k)
        iteration_time = time.time()
        log.append({
            "Time Step": k,
            "timestamp": iteration_time,
            "message": f"Processing item {k} at {iteration_time.isoformat()}"
        })